## Test 1

In [ ]:
from pathlib import Path
from pprint import pprint


def load_fasta(path):
    records = []
    current_record = None

    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            # 跳过空行
            if not line:
                continue

            if line.startswith(">"):
                # 去掉 >，再按 | 拆分标题
                fields = line[1:].split("|")

                current_record = {
                    "accession": fields[0].strip(),
                    "allele": fields[1].strip(),
                    "species": fields[2].strip(),
                    "functionality": fields[3].strip(),
                    "region": fields[4].strip(),
                    "sequence": "",
                }

                records.append(current_record)

            else:
                # 把多行序列连接到当前记录中
                if current_record is None:
                    raise ValueError("文件格式错误：序列前缺少标题")

                current_record["sequence"] += line

    return records

In [6]:
path = "reference.fasta"

# 读取全部记录
records = load_fasta(path)
# 查看读取结果
print("记录总数：", len(records))

if records:
    print("\n第一条记录：")
    pprint(records[0], sort_dicts=False)

记录总数： 17481

第一条记录：
{'accession': 'KT723008',
 'allele': 'IGHA*01',
 'species': 'Bos taurus_Holstein',
 'functionality': 'F',
 'region': 'CH1',
 'sequence': '....XSETSPSIFPLSLGNNDP....AGQVVIGCLVQGFFPS.APLSVTWNQNGDSV....SVRNFPAVLAGS.......LYTMSSQLTLPASLC..PKGQSVTCQVQHL...SKASKTVAVPCIIQ'}


In [7]:
import csv

results = []

for record in records:
    # 1. 筛选人类 TR 的 V 区记录
    if record["species"] != "Homo sapiens":
        continue

    if not record["allele"].startswith("TR"):
        continue

    if record["region"] != "V-REGION":
        continue

    # 2. 获取完整的带缺口序列
    sequence = record["sequence"]

    # 3. 切片，再去掉片段中的点号
    # CDR1：第 27–38 位
    # CDR2：第 56–65 位
    cdr1 = sequence[26:38].replace(".", "")
    cdr2 = sequence[55:65].replace(".", "")

    results.append({
        "allele": record["allele"],
        "cdr1": cdr1,
        "cdr2": cdr2,
    })

if any(row["cdr1"] is Ellipsis or row["cdr2"] is Ellipsis
       for row in results):
    raise ValueError("请先填写 CDR1 和 CDR2 的切片")

output_path = "cdr_sequences.tsv"

with open(output_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["allele", "cdr1", "cdr2"],
        delimiter="\t",
    )
    writer.writeheader()
    writer.writerows(results)

print("输出记录数：", len(results))
print("保存位置：", output_path)

输出记录数： 385
保存位置： cdr_sequences.tsv


## Test 2

In [ ]:
from Bio import Entrez

# 填写你的真实邮箱
Entrez.email = "tan19991103@outlook.com"

query = (
    'medulloblastoma '
    'AND Shih[Author] '
    'AND ("2010/01/01"[Date - Publication] : '
    '"2020/12/31"[Date - Publication])'
)

total = 1000
with Entrez.esearch(
    db="pubmed",
    term=query,
    retmax=total,
) as handle:
    search_result = Entrez.read(handle)

pmids = search_result["IdList"]

print("获取的 PMID 数量：", len(pmids))

获取的 PMID 数量： 40


In [ ]:
# 获取检索结果中的全部编号

with Entrez.efetch(
    db="pubmed",
    id=",".join(pmids),
    retmode="xml",
) as handle:
    papers = Entrez.read(handle)

print("获取的文献数量：", len(papers["PubmedArticle"]))

获取的文献数量： 40


In [13]:
records = []

for paper in papers["PubmedArticle"]:
    citation = paper["MedlineCitation"]
    article = citation["Article"]

    abstract_parts = article.get("Abstract", {}).get("AbstractText", [])

    records.append({
        "pmid": str(citation["PMID"]),
        "title": str(article["ArticleTitle"]),
        "abstract": "\n".join(str(part) for part in abstract_parts),
    })

print("整理后的数量：", len(records))

if records:
    print(records[0])

整理后的数量： 40
{'pmid': '29856958', 'title': 'A Hematogenous Route for Medulloblastoma Leptomeningeal Metastases.', 'abstract': ''}


In [14]:
import csv

with open("papers.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["pmid", "title", "abstract"],
    )
    writer.writeheader()
    writer.writerows(records)

print(f"已保存 {len(records)} 篇文献到 papers.csv")

已保存 40 篇文献到 papers.csv


In [15]:
import csv
import sqlite3

# 从 CSV 读取数据
with open("papers.csv", "r", encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

connection = sqlite3.connect("pubmed_shih_2010_2020.db")

try:
    with connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS papers (
                pmid TEXT PRIMARY KEY,
                title TEXT,
                abstract TEXT
            )
        """)

        connection.executemany("""
            INSERT OR REPLACE INTO papers (pmid, title, abstract)
            VALUES (:pmid, :title, :abstract)
        """, rows)

    count = connection.execute(
        "SELECT COUNT(*) FROM papers"
    ).fetchone()[0]

    print("CSV 记录数：", len(rows))
    print("数据库记录数：", count)

finally:
    connection.close()

CSV 记录数： 40
数据库记录数： 40


In [20]:
import sqlite3
import re

connection = sqlite3.connect("pubmed_shih_2010_2020.db")

try:
    candidates = connection.execute("""
        SELECT pmid, title, abstract
        FROM papers
        WHERE abstract LIKE ?
    """, ("%MYC%",)).fetchall()

finally:
    connection.close()
confirmed_pmids = []

for pmid, title, abstract in candidates:
    if re.search(r"\bMYC\b", abstract, flags=re.IGNORECASE):
        confirmed_pmids.append(pmid)

print("匹配数量：", len(confirmed_pmids))
print("PMID：", confirmed_pmids)

匹配数量： 4
PMID： ['28504719', '25739120', '24493713', '22832581']


In [ ]:
import csv
import sqlite3

if len(confirmed_pmids) != len(set(confirmed_pmids)):
    raise ValueError("PMID 列表有重复")

connection = sqlite3.connect("pubmed_shih_2010_2020.db")

try:
    results = []

    for pmid in confirmed_pmids:
        row = connection.execute(
            "SELECT pmid, title FROM papers WHERE pmid = ?",
            (pmid,),
        ).fetchone()

        if row is None:
            raise ValueError(f"数据库中找不到 PMID：{pmid}")

        results.append(row)

finally:
    connection.close()

with open("myc_papers.tsv", "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file, delimiter="\t")
    writer.writerow(["pmid", "title"])
    writer.writerows(results)

print(f"已导出 {len(results)} 篇文献到 myc_papers.tsv")

已导出 4 篇文献到 myc_papers.tsv


## Test 3 — 统一实验入口
数据读取、清洗、按物种划分、BPE、Dataset/DataLoader、模型和训练均由 `dna_training` 管理。
保留之前的数据划分方法；加权与重采样默认关闭。修改配置后可独立运行本节，无需运行前两题。


In [ ]:
from pathlib import Path
from dna_training import (
    ExperimentConfig, prepare_data, load_prepared, build_dataloaders,
    run_experiment, evaluate_checkpoint, plot_training_curves, compare_experiments,
)
from IPython.display import Image, display

# 另有 joint-mlm.json / weighted.json / resampled.json
config = ExperimentConfig.load("configs/classification.json")
# 也可在此覆盖参数，例如：
# config.training.batch_size = 32
# config.training.max_epochs = 100
# config.training.class_weighting = "inverse_sqrt"
# config.training.sampling = "weighted"
config.validate()
print(config)


### 准备数据和词表
首次生成缓存；配置与原文件未变化时直接复用。BPE 仅使用训练集。
复制整个 `prepared/` 目录即可在另一台机器复用，无需重新训练词表。


In [ ]:
prepared = prepare_data(config.data, config.bpe)
# 也可不读取原 FASTA，直接加载缓存：
# prepared = load_prepared("prepared/datasets/你的数据ID")
train_records = prepared.train_records
val_records = prepared.val_records
test_records = prepared.test_records
tokenizer = prepared.tokenizer
label_to_id = prepared.label_to_id
id_to_label = {v: k for k, v in label_to_id.items()}
print("训练 / 验证 / 测试：", len(train_records), len(val_records), len(test_records))
print("类别数：", len(label_to_id), "词表大小：", len(tokenizer.token_to_id))
print("数据缓存：", prepared.path)
print("数据ID：", prepared.prepared_id)


In [ ]:
example = val_records[0]["sequence"]
print("原序列前80字符：", example[:80])
print("分词前20项：", tokenizer.tokenize(example)[:20])
print("编号前20项：", tokenizer.encode(example)[:20])
assert "".join(tokenizer.tokenize(example)) == tokenizer.normalize(example)

# 只预览。正式训练会新建带固定种子的 loader，预览不影响训练随机状态。
loaders = build_dataloaders(train_records, val_records, tokenizer, label_to_id,
                           config.model, config.training)
batch = next(iter(loaders["train"]))
for name, value in batch.items():
    print(name, tuple(value.shape), value.dtype)
print("类别计数、权重、采样策略：", loaders["balance"])


### 显式启动训练
运行下面单元格才会训练。`fresh` 从头训练；`resume` 从 last.pt 恢复全部状态；
`weights` 只从旧 checkpoint 初始化模型。恢复训练的 max_epochs 是总轮数。
完整设置、数据指纹、类别权重、采样策略、每轮指标和 checkpoint 都写入独立实验目录。


In [ ]:
RUN_MODE = "fresh"  # fresh / resume / weights
CHECKPOINT = None  # resume 或 weights 时填写 checkpoint 路径
if RUN_MODE not in {"fresh", "resume", "weights"}:
    raise ValueError("未知 RUN_MODE")
if RUN_MODE != "fresh" and CHECKPOINT is None:
    raise ValueError("请填写 CHECKPOINT")
result = run_experiment(
    prepared, config,
    resume_from=CHECKPOINT if RUN_MODE == "resume" else None,
    init_from=CHECKPOINT if RUN_MODE == "weights" else None,
)
print("实验目录：", result.run_dir)
print("最佳模型：", result.best_checkpoint)


### 查看训练过程
实时曲线继续使用 TensorBoard：在项目目录启动 `tensorboard --logdir runs --port 6006`。
下面的图片读取已保存的指标；训练进行中重新运行即可更新。服务器也能保存这些图片。


In [ ]:
RUN_DIR = result.run_dir  # 也可改为已有实验目录字符串
curve_path = plot_training_curves(RUN_DIR)
display(Image(filename=str(curve_path)))


### 单独评估
默认评估验证集；明确需要最终测试时才把 split 改为 test。
自动保存指标 JSON、各物种 precision/recall/F1、逐条预测、错误明细、混淆矩阵 CSV/PNG。
混淆矩阵：行=预测，列=真实；颜色按每个真实类别归一化。


In [ ]:
EVAL_CHECKPOINT = result.best_checkpoint  # 或已有 runs/.../checkpoints/best.pt
report = evaluate_checkpoint(prepared, EVAL_CHECKPOINT, split="validation",
                             device=config.training.device)
print(report.metrics)
print("表现最弱的10类：")
for row in sorted(report.per_class, key=lambda r: r["f1-score"])[:10]:
    print(row)
print("报告目录：", report.output_dir)
display(Image(filename=str(report.output_dir / "confusion_matrix.png")))


### 对比实验
先分别运行配置，再填入实验目录。默认检查数据划分与词表相同，
同时输出曲线和最佳验证分类损失对应的指标表。保持相同种子做初步比较，之后可用多种种子验证结论。


In [ ]:
RUNS_TO_COMPARE = []  # 例如 ["runs/classification-...", "runs/weighted-..."]
if RUNS_TO_COMPARE:
    comparison_path = compare_experiments(RUNS_TO_COMPARE, "comparisons/experiments.png")
    display(Image(filename=str(comparison_path)))
else:
    print("填入至少一个已完成实验的目录后运行。")
